# 03 — Regional zooms at native resolution

Surface heat flux structure at the ~2–4 km native LLC2160 resolution in two western
boundary current regions, where mesoscale/submesoscale SST gradients imprint strongly on
air–sea fluxes:

- **Gulf Stream** (82°W–50°W, 25°N–45°N)
- **Kuroshio / Kuroshio Extension** (125°E–165°E, 25°N–45°N)

Faces intersecting each box are plotted directly with `pcolormesh` — no regridding, so the
full native resolution is preserved.

In [ ]:
# Environment check: this notebook must run on SciServer (Kraken domain,
# Oceanography image, "Poseidon DYAMOND (ceph)" data volume), or with
# DYAMOND_ROOT pointing at a local subset.
from dyamond_fluxes import dyamond_root

root = dyamond_root()  # raises with setup instructions if the data volume is absent
print(f"DYAMOND root: {root}")

In [ ]:
from dyamond_fluxes import find_stores_with, nonsolar_flux, open_store, to_positive_down

ocean_store = next(iter(find_stores_with(["oceQnet"])))
ds = open_store(ocean_store)

# A boreal-winter snapshot: strongest latent/sensible heat loss over the western
# boundary currents (cold-air outbreaks).
SNAPSHOT = "2021-01-15T00:00"
snap = ds.sel(time=SNAPSHOT, method="nearest")
print("snapshot:", snap.time.values)

qnet = to_positive_down(snap["oceQnet"])
qsw = to_positive_down(snap["oceQsw"])
qns = nonsolar_flux(qnet, qsw)
if "Depth" in ds:
    mask = ds["Depth"] > 0
    qnet, qns = qnet.where(mask), qns.where(mask)

In [ ]:
from pathlib import Path

from dyamond_fluxes.plotting import plot_region

FIGDIR = Path("../figures")
FIGDIR.mkdir(exist_ok=True)

REGIONS = {
    "gulf_stream": (-82.0, -50.0, 25.0, 45.0),
    "kuroshio": (125.0, 165.0, 25.0, 45.0),
}
lon, lat = ds["XC"], ds["YC"]

In [ ]:
for name, bbox in REGIONS.items():
    fig, ax = plot_region(
        qnet, lon, lat, bbox,
        title=f"Net surface heat flux, {name.replace('_', ' ')}, {str(snap.time.values)[:16]}",
    )
    fig.savefig(FIGDIR / f"qnet_{name}.png", dpi=200, bbox_inches="tight")

In [ ]:
for name, bbox in REGIONS.items():
    fig, ax = plot_region(
        qns, lon, lat, bbox,
        title=f"Non-solar heat flux, {name.replace('_', ' ')}, {str(snap.time.values)[:16]}",
    )
    fig.savefig(FIGDIR / f"qns_{name}.png", dpi=200, bbox_inches="tight")

In winter, the non-solar flux over the Gulf Stream and Kuroshio should show intense ocean
heat loss (large negative $Q_{ns}$, locally beyond $-800$ W m$^{-2}$ during cold-air
outbreaks) organized along the current cores and warm-core rings — structure that only
emerges at kilometer-scale resolution. Compare against a summer snapshot by changing
`SNAPSHOT`.